# Step 3: Load the New Dataset

AD688 Group 3, Module 3. My piece of this is loading the new dataset and getting it into a state the
rest of the team can actually use. PySpark, running on the EC2 box.

Four things happen below: pull the file down, check it before trusting it, load it and save a
verified copy, then compare it against the dataset we used in Step 2.

A note on the industry code. We put the NAICS question to the professor and never got an answer
back, so this stays on 5241, the same cut Step 2 used. Running the identical query against the new
data gives us far more to work with, so there is no reason to move off it. The insurance panel is
saved in the same shape as the Step 2 export, so the two line up column for column.

Nothing here touches Step 2's notebook or files, and nothing here commits or pushes. Run top to
bottom.

## 1. Setup

Imports and paths. Everything hangs off the project root rather than being hardcoded, so this runs
the same on EC2, on a laptop, or from a fresh clone of the team repo.

The big files, the 700 MB raw CSV and the Parquet copy, sit under `data/`. That folder is gitignored
and stays that way: it is the course rule and the data is licensed. Only small outputs go to
`outputs/step3/`.

This cell also sets the keyword net used later to spot data roles, and the list of columns the
project cannot work without. The load stops if any of those are missing.

In [1]:
import csv, sys, os, json
from pathlib import Path

import gdown
from pyspark.sql import SparkSession, functions as F

def find_project_root():
    """Walk up from the current folder until we find the project root (a folder holding
    data/, _quarto.yml or .git). Falls back to the current folder."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data").is_dir() or (p / "_quarto.yml").exists() or (p / ".git").exists():
            return p
    return here

def is_filled(c):
    """True when a column has a real value (not null, not blank)."""
    return F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")

ROOT = find_project_root()

# New dataset: raw download, then a verified Parquet copy for the team. Named for what it
# is (Lightcast job postings) and the year it covers (2024), matching the style of the
# Step 2 folder MET_CareerCompass_2026.
RAW_DIR          = ROOT / "data" / "Lightcast_JobPostings_2024_raw"
RAW_CSV          = RAW_DIR / "lightcast_job_postings.csv"
VERIFIED_PARQUET = ROOT / "data" / "Lightcast_JobPostings_2024" / "lightcast_job_postings.parquet"
# Old dataset from Step 2 (read only, for the comparison).
OLD_DIR          = ROOT / "data" / "MET_CareerCompass_2026"
# Small outputs that are safe to share.
OUT_DIR          = ROOT / "outputs" / "step3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The only source for the new data, from the M03_Proj instructions.
DATASET_ID    = "1V2GCHGt2dkFGqVBeoUFckU4IhUgk4ocQ"
MIN_RAW_BYTES = 100 * 1024 * 1024   # a real copy is ~700 MB; anything under 100 MB is not the data

# Columns the project needs. The load stops if any is missing.
REQUIRED_COLUMNS = ["ID", "POSTED", "TITLE_RAW", "NAICS_2022_4",
                    "SALARY_FROM", "SALARY_TO", "MIN_YEARS_EXPERIENCE"]

# The team's data-role keyword net (matched on the lower-cased raw title), reused as planned.
# Note: the Step 2 flag also matched " bi " (BI as a separate word). This planned net leaves it
# out. On the new data that term would add 11 postings (621 instead of 610); on the old data it
# adds none.
NET_REGEX = ("data engineer|analytics engineer|etl|data pipeline|data platform|big data|"
             "ml engineer|machine learning engineer|data architect|database|data scientist|"
             "data science|data analyst|business intelligence|data governance|data management")
# Strict flag: the title itself says data engineer or analytics engineer.
STRICT_REGEX = "data engineer|analytics engineer"

print("Project root:", ROOT)
print("Raw CSV     :", RAW_CSV)
print("Parquet copy:", VERIFIED_PARQUET)
print("Outputs     :", OUT_DIR)

Project root: /home/ubuntu/ad688
Raw CSV     : /home/ubuntu/ad688/data/Lightcast_JobPostings_2024_raw/lightcast_job_postings.csv
Parquet copy: /home/ubuntu/ad688/data/Lightcast_JobPostings_2024/lightcast_job_postings.parquet
Outputs     : /home/ubuntu/ad688/outputs/step3


## 2. Download the new dataset

Pulls the file from Google Drive with `gdown`. It is about 700 MB, so the cell looks for a full-size
copy first and skips the download when one is already there. No sense pulling that down on every
run.

In [2]:
# Download only if we do not already have a full-size copy.
RAW_DIR.mkdir(parents=True, exist_ok=True)
if RAW_CSV.exists() and RAW_CSV.stat().st_size > MIN_RAW_BYTES:
    print("Already downloaded, skipping the download.")
else:
    gdown.download(id=DATASET_ID, output=str(RAW_CSV), quiet=False)

raw_bytes = RAW_CSV.stat().st_size
print("File:", RAW_CSV.name)
print("Size:", f"{raw_bytes:,} bytes ({raw_bytes / 1024**2:,.0f} MB)")

Already downloaded, skipping the download.
File: lightcast_job_postings.csv
Size: 716,669,689 bytes (683 MB)


## 3. Check the file before loading it

Downloads fail quietly. You can end up with an HTML error page sitting there under a `.csv` name and
not find out until something strange happens three cells later. So nothing gets loaded until it
clears four checks:

1. Is it actually a CSV? Decided by reading the first bytes, not by trusting the file extension.
2. Is it big enough? Anything under 100 MB is not the real dataset.
3. How many rows are really in it? This is the one that catches people out. Job descriptions contain
   line breaks inside the quoted text, so counting lines gives about 13 million against 72,498 real
   rows. A proper CSV parser is the only way to get the true number.
4. Are the columns the project needs all there?

The row count from this cell gets reused further down to verify the Spark load, so it is worth
getting right rather than eyeballing.

In [3]:
# 1) Format by file bytes, not by extension.
with open(RAW_CSV, "rb") as f:
    head = f.read(4096)

signatures = {b"PAR1": "parquet", b"PK\x03\x04": "zip/xlsx", b"\x1f\x8b": "gzip"}
detected = "text"
for sig, name in signatures.items():
    if head.startswith(sig):
        detected = name

has_bom     = head.startswith(b"\xef\xbb\xbf")            # byte-order mark at the very start
first_line  = head.decode("utf-8-sig", errors="ignore").splitlines()[0]
looks_csv   = detected == "text" and first_line.startswith("ID,") and first_line.count(",") > 50
print("Detected format from bytes:", "csv" if looks_csv else detected)
print("Starts with a byte-order mark (BOM):", has_bom)
assert looks_csv, "The file does not look like the expected CSV. Do not load it."

# 2) Size check.
assert raw_bytes > MIN_RAW_BYTES, "File is too small to be the real dataset."
print("Size check passed:", f"{raw_bytes / 1024**2:,.0f} MB")

Detected format from bytes: csv
Starts with a byte-order mark (BOM): True
Size check passed: 683 MB


In [4]:
# 3) Independent row count with Python's CSV parser (streams the file, low memory).
#    utf-8-sig strips the BOM. The field-size limit is raised because job descriptions are long.
csv.field_size_limit(2**31 - 1)

csv_rows, bad_rows, blank_id_rows, empty_rows, ids = 0, 0, 0, 0, set()
with open(RAW_CSV, newline="", encoding="utf-8-sig") as f:
    reader = csv.reader(f)
    header = next(reader)
    n_cols = len(header)
    id_pos = header.index("ID")
    for row in reader:
        csv_rows += 1
        if len(row) != n_cols:
            bad_rows += 1          # a row with the wrong number of fields would mean a broken parse
            continue
        if row[id_pos].strip() == "":
            blank_id_rows += 1     # some source rows have no ID; we count them, we do not hide them
            if not any(v.strip() for v in row):
                empty_rows += 1    # ...and some of those have no value in any column
        else:
            ids.add(row[id_pos])

# For contrast: a plain line count (this is the number NOT to use).
with open(RAW_CSV, "rb") as f:
    plain_lines = sum(chunk.count(b"\n") for chunk in iter(lambda: f.read(1 << 20), b"")) - 1

print("Columns in header          :", n_cols)
print("Rows (CSV parser)          :", f"{csv_rows:,}")
print("Plain line count minus head:", f"{plain_lines:,}   (over-counts, because of line breaks inside quotes)")
print("Rows with wrong field count:", bad_rows)
print("Rows with a blank ID       :", blank_id_rows, f"(of which entirely empty: {empty_rows})")
print("Unique non-blank IDs       :", f"{len(ids):,}")
print("Duplicate non-blank IDs    :", (csv_rows - bad_rows - blank_id_rows) - len(ids))

# 4) Required columns.
missing = [c for c in REQUIRED_COLUMNS if c not in header]
print("Required columns missing   :", missing if missing else "none")
assert not missing, f"Missing required columns: {missing}"
assert bad_rows == 0, "Some rows have the wrong number of fields."

Columns in header          : 131
Rows (CSV parser)          : 72,498
Plain line count minus head: 12,955,714   (over-counts, because of line breaks inside quotes)
Rows with wrong field count: 0
Rows with a blank ID       : 22 (of which entirely empty: 22)
Unique non-blank IDs       : 72,476
Duplicate non-blank IDs    : 0
Required columns missing   : none


## 4. Load with Spark

Three options matter here, and getting any of them wrong corrupts the load without throwing an
error:

- `multiLine` has to be on, same line-break reason as above. Without it Spark splits rows in the
  middle of job descriptions.
- Quote and escape are both `"`, which is how this file writes a quote inside a quoted field.
- The file opens with a byte-order mark, so the first column name arrives with an invisible
  character attached. Stripping it means the column is `ID` rather than something that looks like
  `ID` on screen but matches nothing.

Parquet row groups are set deliberately small. This instance has 908 MB of RAM and the default sizes
do not fit in it.

In [5]:
spark = (SparkSession.builder
         .appName("AD688-Step3-LoadDataset")
         .master("local[1]")
         .config("spark.driver.memory", "512m")                            # small instance, keep it small
         .config("spark.ui.showConsoleProgress", "false")                  # keep notebook output clean
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.hadoop.parquet.block.size", str(32 * 1024 * 1024)) # smaller row groups = less memory
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

raw = (spark.read
       .option("header", True)
       .option("multiLine", True)            # line breaks inside quoted fields
       .option("quote", '"')
       .option("escape", '"')                # a doubled quote inside a field means one literal quote
       .option("encoding", "UTF-8")
       .option("maxCharsPerColumn", -1)      # job descriptions can be long
       .csv(str(RAW_CSV)))

# Strip a leading byte-order mark from column names if Spark left one in.
raw = raw.toDF(*[c.lstrip("\ufeff").strip() for c in raw.columns])
assert raw.columns[0] == "ID", f"First column is {raw.columns[0]!r}, expected 'ID'"
print("Columns read by Spark:", len(raw.columns), "| first column:", raw.columns[0])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/ubuntu/ad688/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/21 15:13:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Columns read by Spark: 131 | first column: ID


## 5. Save a verified copy, then describe it

Writes the data out to Parquet. Smaller and faster to read than the CSV, and the same format the
Step 2 dataset came in.

The write is skipped when a finished copy is already in place, because parsing 700 MB of CSV is by
far the slowest thing in this notebook. The checks underneath still run every time: the copy is read
back and its row count, blank IDs and unique IDs are compared against what the CSV parser found in
section 3. If any of those disagree the notebook stops rather than quietly carrying on with a
partial copy.

The Parquet copy lives in `data/Lightcast_JobPostings_2024/` and is gitignored.

The cell after that builds a data dictionary: every column, a plain-English grouping, and how often
each one is actually filled in. It counts in small batches, because doing all 131 columns in one
pass, with the full job description text among them, runs this machine out of memory.

In [6]:
# Write the Parquet copy (Spark parses the whole CSV once here).
# Skipped when a completed copy is already in place, the same way the download is skipped above.
# The checks below still run on every pass, so the copy is re-verified against the source
# whether it was just written or not.
if (VERIFIED_PARQUET / "_SUCCESS").exists():
    print("Parquet copy already in place, skipping the write.")
else:
    raw.write.mode("overwrite").parquet(str(VERIFIED_PARQUET))

# Read the copy back and verify it against the independent CSV-parser results.
new = spark.read.parquet(str(VERIFIED_PARQUET))
spark_rows = new.count()

print("Rows: CSV parser =", f"{csv_rows:,}", "| Spark (Parquet copy) =", f"{spark_rows:,}")
assert spark_rows == csv_rows, "Spark row count does not match the CSV parser. Do not use this copy."
assert len(new.columns) == n_cols, "Column count changed during the load."

blank_rows_df    = new.filter(~is_filled("ID"))
blank_ids        = blank_rows_df.count()
spark_unique_ids = new.filter(is_filled("ID")).select("ID").distinct().count()
print("Blank-ID rows      : Spark =", blank_ids, "| CSV parser =", blank_id_rows)
print("Unique non-blank IDs: Spark =", f"{spark_unique_ids:,}", "| CSV parser =", f"{len(ids):,}")
assert blank_ids == blank_id_rows and spark_unique_ids == len(ids), "Spark and the CSV parser disagree on IDs."

# What do the blank-ID rows contain? (counts only)
blank_profile = blank_rows_df.agg(
    F.sum(F.when(is_filled("POSTED"), 1).otherwise(0)).alias("POSTED filled"),
    F.sum(F.when(is_filled("NAICS_2022_4"), 1).otherwise(0)).alias("NAICS_2022_4 filled"),
    F.sum(F.when(is_filled("TITLE_RAW"), 1).otherwise(0)).alias("TITLE_RAW filled"),
).collect()[0].asDict()
print("Among the blank-ID rows:", {k: int(v or 0) for k, v in blank_profile.items()})
print("Verified: the Parquet copy matches the source file.")

Parquet copy already in place, skipping the write.
Rows: CSV parser = 72,498 | Spark (Parquet copy) = 72,498
Blank-ID rows      : Spark = 22 | CSV parser = 22
Unique non-blank IDs: Spark = 72,476 | CSV parser = 72,476
Among the blank-ID rows: {'POSTED filled': 0, 'NAICS_2022_4 filled': 0, 'TITLE_RAW filled': 0}
Verified: the Parquet copy matches the source file.


In [7]:
# Data dictionary: every column, a plain-English group, and how many rows have a value.
def column_group(name):
    """Rough plain-English group for a column, based on its name."""
    rules = [
        (("MODELED_", "LAST_UPDATED", "DUPLICATES", "POSTED", "EXPIRED", "DURATION",
          "SOURCE", "URL", "ACTIVE_", "ID"), "Posting record"),
        (("BODY",), "Job description text"),
        (("TITLE",), "Job title"),
        (("COMPANY",), "Employer"),
        (("EDUCATION", "MIN_EDULEVELS", "MAX_EDULEVELS", "CIP"), "Education"),
        (("EMPLOYMENT_TYPE", "MIN_YEARS", "MAX_YEARS", "IS_INTERNSHIP"), "Job type and experience"),
        (("SALARY", "ORIGINAL_PAY_PERIOD"), "Pay"),
        (("REMOTE_TYPE",), "Remote work"),
        (("LOCATION", "CITY", "COUNTY", "MSA", "STATE"), "Location"),
        (("NAICS",), "Industry code (NAICS)"),
        (("ONET", "SOC"), "Occupation code (O*NET, SOC)"),
        (("SKILLS", "SPECIALIZED_SKILLS", "COMMON_SKILLS", "SOFTWARE_SKILLS", "CERTIFICATIONS"), "Skills and certifications"),
        (("LIGHTCAST_SECTORS",), "Lightcast sector"),
    ]
    for prefixes, label in rules:
        if name.startswith(prefixes):
            return label
    return "Other"

# Count filled values in small batches of columns. One pass over all 131 columns at once
# (including the very long BODY text) ran this small instance out of memory, so the long
# text column goes on its own and the rest go 10 at a time.
def count_filled(cols):
    return new.agg(*[F.sum(F.when(is_filled(c), 1).otherwise(0)).alias(c) for c in cols]).collect()[0].asDict()

heavy = [c for c in ["BODY"] if c in new.columns]
rest  = [c for c in new.columns if c not in heavy]
counts = {}
for cols in [heavy] * bool(heavy) + [rest[i:i + 10] for i in range(0, len(rest), 10)]:
    counts.update(count_filled(cols))

dictionary = [
    {"position": i + 1, "column": c, "group": column_group(c),
     "type_in_parquet": dict(new.dtypes)[c],
     "rows_filled": int(counts[c]), "percent_filled": round(100 * counts[c] / spark_rows, 1)}
    for i, c in enumerate(new.columns)
]

dict_path = OUT_DIR / "data_dictionary.csv"
with open(dict_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(dictionary[0].keys()))
    w.writeheader()
    w.writerows(dictionary)

print("Saved:", dict_path, f"({len(dictionary)} columns)")
print("Required columns, share of rows filled:")
for row in dictionary:
    if row["column"] in REQUIRED_COLUMNS + ["SKILLS_NAME", "BODY"]:
        print(f"  {row['column']:<22} {row['percent_filled']:>5}%")

Saved: /home/ubuntu/ad688/outputs/step3/data_dictionary.csv (131 columns)
Required columns, share of rows filled:
  ID                     100.0%
  POSTED                 100.0%
  TITLE_RAW               99.9%
  BODY                    99.9%
  MIN_YEARS_EXPERIENCE    68.1%
  SALARY_TO               44.7%
  SALARY_FROM             44.7%
  SKILLS_NAME             99.9%
  NAICS_2022_4            99.9%


## 6. Old dataset versus new dataset

Same rules on both: same industry code, same keyword net, same strict data engineer test. Counts
only here, no rows get saved.

"Salary present" means `SALARY_FROM` has something in it. "Skills populated" means `SKILLS_NAME` is
not blank.

Worth noting which keyword net this uses. It is the planned one, which leaves out `" bi "` as a
standalone word. Step 2's flag included it. On the new data that single term pulls in 11 extra
postings, 621 instead of 610. On the old data it changes nothing.

In [8]:
def parse_posted():
    """POSTED as a real date. Handles the new file's M/d/yyyy and the old file's yyyy-MM-dd."""
    return F.coalesce(
        F.expr("CAST(try_to_timestamp(CAST(POSTED AS STRING), 'M/d/yyyy') AS DATE)"),
        F.expr("CAST(try_to_timestamp(SUBSTR(CAST(POSTED AS STRING), 1, 10), 'yyyy-MM-dd') AS DATE)"),
    )

def profile(df):
    """Counts for one dataset, all in a single pass."""
    title      = F.lower(F.col("TITLE_RAW"))
    in_net     = F.coalesce(title.rlike(NET_REGEX), F.lit(False))
    is_strict  = F.coalesce(title.rlike(STRICT_REGEX), F.lit(False))
    is_ins     = F.col("NAICS_2022_4").cast("string") == "5241"
    one_if     = lambda cond: F.sum(F.when(cond, 1).otherwise(0))
    posted     = parse_posted()
    r = df.agg(
        F.count(F.lit(1)).alias("total_rows"),
        one_if(is_ins).alias("insurance_5241_rows"),
        one_if(is_ins & in_net).alias("insurance_data_role_rows"),
        one_if(is_ins & is_strict).alias("insurance_strict_data_engineer_rows"),
        one_if(is_ins & in_net & is_filled("SALARY_FROM")).alias("data_role_with_salary"),
        one_if(is_ins & in_net & is_filled("MIN_YEARS_EXPERIENCE")).alias("data_role_with_min_years_experience"),
        one_if(is_ins & is_filled("SKILLS_NAME")).alias("insurance_with_skills"),
        F.min(posted).alias("posted_first"),
        F.max(posted).alias("posted_last"),
        one_if(F.col("POSTED").isNotNull() & posted.isNull()).alias("posted_unreadable"),
    ).collect()[0].asDict()
    return {k: (str(v) if k.startswith("posted_") and k != "posted_unreadable" else int(v)) for k, v in r.items()}

old_files = sorted(str(p) for p in OLD_DIR.glob("*.parquet"))
old = profile(spark.read.parquet(*old_files))
newp = profile(new)

labels = [
    ("total_rows",                          "Total postings"),
    ("insurance_5241_rows",                 "Insurance (NAICS 5241) postings"),
    ("insurance_data_role_rows",            "  of those, matching the data-role keyword net"),
    ("insurance_strict_data_engineer_rows", "  of those, strict data engineer / analytics engineer titles"),
    ("data_role_with_salary",               "  data-role postings with a salary (SALARY_FROM)"),
    ("data_role_with_min_years_experience", "  data-role postings with minimum years of experience"),
    ("insurance_with_skills",               "Insurance postings with SKILLS_NAME filled"),
    ("posted_first",                        "First POSTED date"),
    ("posted_last",                         "Last POSTED date"),
    ("posted_unreadable",                   "POSTED values that could not be read as a date"),
]

def fmt(v):
    return f"{v:,}" if isinstance(v, int) else v

print(f"{'Measure':<62}{'OLD (Step 2)':>16}{'NEW (Step 3)':>16}")
print("-" * 94)
for key, label in labels:
    print(f"{label:<62}{fmt(old[key]):>16}{fmt(newp[key]):>16}")

# Save the comparison (aggregate numbers only, no posting rows).
cmp_csv = OUT_DIR / "comparison_old_vs_new.csv"
with open(cmp_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["measure", "old_step2_dataset", "new_step3_dataset"])
    for key, label in labels:
        w.writerow([label.strip(), old[key], newp[key]])
print("\nSaved:", cmp_csv)

Measure                                                           OLD (Step 2)    NEW (Step 3)
----------------------------------------------------------------------------------------------
Total postings                                                         165,386          72,498
Insurance (NAICS 5241) postings                                          1,895           1,908
  of those, matching the data-role keyword net                              27             610
  of those, strict data engineer / analytics engineer titles                 6              29
  data-role postings with a salary (SALARY_FROM)                             5             405
  data-role postings with minimum years of experience                        0             521
Insurance postings with SKILLS_NAME filled                               1,271           1,908
First POSTED date                                                   2026-01-02      2024-05-01
Last POSTED date                                  

In [9]:
# Check the results against the numbers the team planned for. Any mismatch stops the notebook.
planned = [
    ("New total rows",                              72_498,       newp["total_rows"]),
    ("Old total rows",                              165_386,      old["total_rows"]),
    ("New Insurance (5241) rows",                   1_908,        newp["insurance_5241_rows"]),
    ("New 5241 x keyword net",                      610,          newp["insurance_data_role_rows"]),
    ("Old 5241 x keyword net",                      27,           old["insurance_data_role_rows"]),
    ("New strict data engineer titles",             29,           newp["insurance_strict_data_engineer_rows"]),
    ("Old strict data engineer titles",             6,            old["insurance_strict_data_engineer_rows"]),
    ("New data-role rows with salary",              405,          newp["data_role_with_salary"]),
    ("New Insurance rows with SKILLS_NAME filled",  1_908,        newp["insurance_with_skills"]),
    ("New first POSTED date",                       "2024-05-01", newp["posted_first"]),
    # The planning notes said the range ended 9/9/2024. That is the last value when POSTED is
    # sorted as TEXT ("9/9/2024" sorts after "9/30/2024"). As real dates the last posting is
    # 2024-09-30, so that is the value we check against (see the text-versus-date lines below).
    ("New last POSTED date (as real dates)",        "2024-09-30", newp["posted_last"]),
]
all_ok = True
for name, expected, actual in planned:
    ok = expected == actual
    all_ok &= ok
    print(f"{'PASS' if ok else 'MISMATCH':<9}{name:<46} expected {fmt(expected):>12}   got {fmt(actual):>12}")
assert all_ok, "At least one number does not match the plan. Stop and report it before using this data."
print("\nAll planned numbers reproduced.")

# Why the last POSTED date is 2024-09-30 and not 9/9/2024: sorting the text is misleading.
text_max = new.agg(F.max("POSTED")).collect()[0][0]
print(f"\nPOSTED sorted as text gives a last value of {text_max!r} (misleading: '9/9/2024' sorts after '9/30/2024').")
print(f"POSTED read as real dates gives a last value of {newp['posted_last']}.")

PASS     New total rows                                 expected       72,498   got       72,498
PASS     Old total rows                                 expected      165,386   got      165,386
PASS     New Insurance (5241) rows                      expected        1,908   got        1,908
PASS     New 5241 x keyword net                         expected          610   got          610
PASS     Old 5241 x keyword net                         expected           27   got           27
PASS     New strict data engineer titles                expected           29   got           29
PASS     Old strict data engineer titles                expected            6   got            6
PASS     New data-role rows with salary                 expected          405   got          405
PASS     New Insurance rows with SKILLS_NAME filled     expected        1,908   got        1,908
PASS     New first POSTED date                          expected   2024-05-01   got   2024-05-01
PASS     New last POSTED date 

### What the numbers say

Same filter, new data, and the thin-data problem from Step 2 is gone. Step 2 gave 27 matching
postings with a salary on 5 of them, which is not enough to build anything on. The new data gives
hundreds, with salary on most of them.

The catch is the window. The postings moved from 2026 back to 2024, so this is a different period
rather than a bigger sample of the same one. That needs saying out loud any time these numbers get
quoted.

In [10]:
print(f"Data-role postings in Insurance : {old['insurance_data_role_rows']:,} (old)  ->  {newp['insurance_data_role_rows']:,} (new)")
print(f"Strict data engineer titles      : {old['insurance_strict_data_engineer_rows']:,} (old)  ->  {newp['insurance_strict_data_engineer_rows']:,} (new)")
print(f"Data-role postings with a salary : {old['data_role_with_salary']:,} of {old['insurance_data_role_rows']:,} (old)  ->  {newp['data_role_with_salary']:,} of {newp['insurance_data_role_rows']:,} (new)")
print(f"Posting dates                    : {old['posted_first']} to {old['posted_last']} (old)  ->  {newp['posted_first']} to {newp['posted_last']} (new)")

Data-role postings in Insurance : 27 (old)  ->  610 (new)
Strict data engineer titles      : 6 (old)  ->  29 (new)
Data-role postings with a salary : 5 of 27 (old)  ->  405 of 610 (new)
Posting dates                    : 2026-01-02 to 2026-07-21 (old)  ->  2024-05-01 to 2024-09-30 (new)


## 7. Save the insurance panel

Step 2 produced `insurance_5241_all_postings.csv` and `.xlsx`: every insurance posting, 34 columns,
with `IS_DATA_ROLE_MATCH` sitting directly after `TITLE_CLEAN`. This builds the same thing from the
new data, same columns in the same order, written the same way, so the two can be lined up column
for column. The file carries a `_step3` suffix so it is obvious which step it came from once it
leaves this folder.

The flag is the part to pay attention to. It uses Step 2's regex exactly, `" bi "` included. A column
with the same name in two files has to mean the same thing in both, otherwise the name lies. The
price of that is this file flags 621 rows while section 6 reports 610, since section 6 uses the
planned net without that term. Both numbers are right. They answer different questions.

In [11]:
# Save the Insurance panel in the same shape as the Step 2 export.
#
# The flag uses the Step 2 regex exactly, including " bi " (BI as a separate word), so that
# IS_DATA_ROLE_MATCH means the same thing in both files. It is why this file flags 621 rows
# while the comparison above reports 610, which uses the planned net without " bi ".
STEP2_FLAG_REGEX = ("data engineer|analytics engineer|etl|data pipeline|data platform|big data|"
                    "ml engineer|machine learning engineer|data architect|database|data scientist|"
                    "data science|data analyst|business intelligence| bi |data governance|"
                    "data management")

# The Step 2 export's 34 columns, in the Step 2 order. IS_DATA_ROLE_MATCH is the only derived
# column; every other one is an original Lightcast field.
STEP2_COLUMNS = ["ID", "POSTED", "EXPIRED", "DURATION", "URL", "TITLE_RAW", "TITLE_NAME",
                 "TITLE_CLEAN", "IS_DATA_ROLE_MATCH", "COMPANY_NAME", "COMPANY_IS_STAFFING",
                 "EMPLOYMENT_TYPE_NAME", "MIN_YEARS_EXPERIENCE", "MAX_YEARS_EXPERIENCE",
                 "IS_INTERNSHIP", "MIN_EDULEVELS_NAME", "MAX_EDULEVELS_NAME", "SALARY",
                 "SALARY_FROM", "SALARY_TO", "ORIGINAL_PAY_PERIOD", "REMOTE_TYPE_NAME",
                 "LOCATION", "CITY_NAME", "STATE_NAME", "MSA_NAME", "NAICS_2022_4",
                 "SOC_2021_4_NAME", "ONET_NAME", "SKILLS_NAME", "SPECIALIZED_SKILLS_NAME",
                 "COMMON_SKILLS_NAME", "SOFTWARE_SKILLS_NAME", "CERTIFICATIONS_NAME"]

source_cols = [c for c in STEP2_COLUMNS if c != "IS_DATA_ROLE_MATCH"]
missing_cols = [c for c in source_cols if c not in new.columns]
assert not missing_cols, f"The new dataset is missing Step 2 columns: {missing_cols}"

insurance = (new
             .filter(F.col("NAICS_2022_4").cast("string") == "5241")
             .withColumn("IS_DATA_ROLE_MATCH",
                         F.when(F.lower(F.col("TITLE_RAW")).rlike(STEP2_FLAG_REGEX), 1).otherwise(0))
             .select(*STEP2_COLUMNS))

# Same writer as Step 2 (pandas), so the two files are produced identically. The panel is
# small enough for this to be safe: about 1,900 rows.
pdf       = insurance.toPandas()
xlsx_path = OUT_DIR / "insurance_5241_all_postings_step3.xlsx"
csv_path  = OUT_DIR / "insurance_5241_all_postings_step3.csv"
pdf.to_excel(xlsx_path, index=False)
pdf.to_csv(csv_path, index=False)

flagged = int(pdf["IS_DATA_ROLE_MATCH"].sum())
print("Rows exported     :", f"{len(pdf):,}")
print("Columns           :", len(pdf.columns))
print("Flagged as 1      :", flagged, "(Step 2 regex, includes ' bi ')")
print("For contrast, the planned net without ' bi ' matched:", newp["insurance_data_role_rows"])
print("Excel:", xlsx_path)
print("CSV  :", csv_path)

# The panel must hold every Insurance posting counted in section 6, in the Step 2 shape.
assert len(pdf) == newp["insurance_5241_rows"], "Exported rows do not match the Insurance count above."
assert list(pdf.columns) == STEP2_COLUMNS, "Column order does not match the Step 2 export."
assert len(STEP2_COLUMNS) == 34, "The Step 2 export has 34 columns."
assert flagged == 621, f"Expected 621 flagged rows with the Step 2 regex, got {flagged}."
print("\nMatches the Step 2 export: 34 columns, same order, same flag definition.")

/home/ubuntu/ad688/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/ubuntu/ad688/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Rows exported     : 1,908
Columns           : 34
Flagged as 1      : 621 (Step 2 regex, includes ' bi ')
For contrast, the planned net without ' bi ' matched: 610
Excel: /home/ubuntu/ad688/outputs/step3/insurance_5241_all_postings_step3.xlsx
CSV  : /home/ubuntu/ad688/outputs/step3/insurance_5241_all_postings_step3.csv

Matches the Step 2 export: 34 columns, same order, same flag definition.


## 8. Step 2 export versus Step 3 export

Section 6 compared the two raw datasets. This compares the two finished export files, which is the
narrower and more useful question, since those are the files the rest of the project actually
touches.

| File | What it compares | Scale |
|---|---|---|
| `comparison_old_vs_new.csv` | the raw datasets, all industries | 165,386 against 72,498 postings |
| `variance_analysis_step2_vs_step3.csv` | the two insurance exports | 1,895 against 1,908 rows |

Both exports use the same flag definition, so the flagged counts here are directly comparable. No
Spark in this cell, it just reads the two CSVs off disk, so it runs in about a second.

Two of the measures come out odd: the company count drops hard, and Step 2 reports more distinct
states than there are states. Both are marked unexplained rather than written up as findings,
because neither has been chased down yet.

In [12]:
# Variance analysis: the Step 2 export against the Step 3 export, file to file.
# Pure Python, no Spark needed, since both are finished CSVs on disk.
from datetime import datetime

STEP2_EXPORT = ROOT / "outputs" / "step2" / "insurance_5241_all_postings.csv"
STEP3_EXPORT = OUT_DIR / "insurance_5241_all_postings_step3.csv"

def parse_posted_text(s):
    """POSTED as a real date. Step 2's file is yyyy-MM-dd, Step 3's is M/d/yyyy."""
    s = (s or "").strip()
    for fmt in ("%m/%d/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s[:10], fmt).date()
        except ValueError:
            pass
    return None

def export_profile(path):
    """Everything we want to compare, in one pass over a finished export."""
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    filled   = lambda c: sum(1 for r in rows if (r.get(c) or "").strip())
    distinct = lambda c: len({(r.get(c) or "").strip() for r in rows if (r.get(c) or "").strip()})
    flagged  = [r for r in rows if r.get("IS_DATA_ROLE_MATCH") == "1"]
    dates    = [d for d in (parse_posted_text(r.get("POSTED")) for r in rows) if d]
    return {
        "rows": len(rows),
        "cols": len(rows[0]),
        "flagged": len(flagged),
        "salary": filled("SALARY_FROM"),
        "flagged_salary": sum(1 for r in flagged if (r.get("SALARY_FROM") or "").strip()),
        "minyears": filled("MIN_YEARS_EXPERIENCE"),
        "skills": filled("SKILLS_NAME"),
        "companies": distinct("COMPANY_NAME"),
        "states": distinct("STATE_NAME"),
        "first": min(dates).isoformat() if dates else "n/a",
        "last": max(dates).isoformat() if dates else "n/a",
    }

s2, s3 = export_profile(STEP2_EXPORT), export_profile(STEP3_EXPORT)

# (label, key, kind, note). "count" gets a difference and a percent change, "date" does not.
MEASURES = [
    ("Rows in the export", "rows", "count",
     "Every Insurance NAICS 5241 posting in that step."),
    ("Columns", "cols", "count",
     "Identical by design: the same 34 columns in the same order."),
    ("Flagged IS_DATA_ROLE_MATCH", "flagged", "count",
     "Same regex in both files, including the ' bi ' term, so these are directly comparable."),
    ("Rows with SALARY_FROM", "salary", "count",
     "Salary present anywhere in the Insurance panel."),
    ("Flagged rows with SALARY_FROM", "flagged_salary", "count",
     "The number that matters: data-role postings we can actually analyse pay for."),
    ("Rows with MIN_YEARS_EXPERIENCE", "minyears", "count",
     "Empty in Step 2, so experience analysis was impossible on the old data."),
    ("Rows with SKILLS_NAME", "skills", "count",
     "Fully populated in Step 3."),
    ("Distinct COMPANY_NAME", "companies", "count",
     "UNEXPLAINED: Step 3 has more rows but far fewer employers, so the new data is more "
     "concentrated. Check before describing either as 'the market'."),
    ("Distinct STATE_NAME", "states", "count",
     "UNEXPLAINED: 51 is clean (states plus DC), 97 is not a real state count, so Step 2's "
     "field probably holds combined or multi-value entries. Check before mapping it."),
    ("First posting date", "first", "date",
     "Different time window, not just a bigger sample."),
    ("Last posting date", "last", "date",
     "Different time window, not just a bigger sample."),
]

def variance_row(label, key, kind, note):
    a, b = s2[key], s3[key]
    if kind == "date":
        return [label, a, b, "n/a", "n/a", note]
    diff = b - a
    if a == 0:
        change = "n/a (Step 2 is zero)"
    else:
        change = f"{(b - a) / a * 100:+.1f}%"
    share = ""
    if key in ("flagged", "salary", "minyears", "skills"):
        share = f" Share of rows: {a / s2['rows'] * 100:.1f}% to {b / s3['rows'] * 100:.1f}%."
    return [label, a, b, f"{diff:+,}", change, note + share]

variance = [variance_row(*m) for m in MEASURES]

var_path = OUT_DIR / "variance_analysis_step2_vs_step3.csv"
with open(var_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["measure", "step2_export", "step3_export", "difference", "percent_change", "note"])
    w.writerows(variance)

print(f"{'Measure':<34}{'STEP 2':>14}{'STEP 3':>14}{'DIFF':>12}")
print("-" * 74)
for label, a, b, diff, change, _ in variance:
    print(f"{label:<34}{str(a):>14}{str(b):>14}{diff:>12}")
print("\nSaved:", var_path)

# Both files must have the identical schema, or this comparison is not like for like.
assert s2["cols"] == s3["cols"] == 34, "The two exports no longer share the 34-column schema."

Measure                                   STEP 2        STEP 3        DIFF
--------------------------------------------------------------------------
Rows in the export                          1895          1908         +13
Columns                                       34            34          +0
Flagged IS_DATA_ROLE_MATCH                    27           621        +594
Rows with SALARY_FROM                        472          1281        +809
Flagged rows with SALARY_FROM                  5           416        +411
Rows with MIN_YEARS_EXPERIENCE                 0          1622      +1,622
Rows with SKILLS_NAME                       1271          1908        +637
Distinct COMPANY_NAME                        873           126        -747
Distinct STATE_NAME                           97            51         -46
First posting date                    2026-02-04    2024-05-01         n/a
Last posting date                     2026-07-17    2024-09-30         n/a

Saved: /home/ubuntu/ad68

## 9. What this run produced

- `outputs/step3/insurance_5241_all_postings_step3.csv` and `.xlsx`, 1,908 insurance postings in the
  same 34 columns as Step 2. This is the file the rest of the project works from.
- A verified Parquet copy in `data/Lightcast_JobPostings_2024/`, gitignored, never shared through
  git. Named for what the data is and the year it covers, the same way `MET_CareerCompass_2026` is.
- `outputs/step3/data_dictionary.csv`, all 131 columns and how often each is filled in.
- `outputs/step3/comparison_old_vs_new.csv`, the raw dataset counts from section 6.
- `outputs/step3/variance_analysis_step2_vs_step3.csv`, the export against export comparison from
  section 8.

One data note and one decision.

The source file has 22 rows with no ID, and those same rows have no date, industry code or title
either. Spark and the CSV parser agree on that, so it comes from the source and is not something the
load did. They are kept so the row count matches the source, and they do not affect any number
above.

On the industry code: we asked the professor and never heard back, so this stays on 5241, the same
cut as Step 2. Same query, new data, much stronger results. Keeping it is the easy call.

Spark gets stopped below to hand the memory back.

In [13]:
# Release Spark's memory now that we're done with it.
spark.stop()
print("Spark stopped.")

Spark stopped.
